In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import random

In [7]:
dataset = pd.read_csv('dataset_30000.csv')
def weighted_snacks(goal):
    if goal == "loss":
        return np.random.choice([0, 1], p=[0.7, 0.3])
    if goal == "maintain":
        return np.random.choice([1, 2], p=[0.9, 0.1])
    if goal == "gain":
        return np.random.choice([1, 2], p=[0.2, 0.8])
    return 1 

def random_meals(goal):
    if goal == "loss":
        return np.random.choice([3, 4], p=[0.7, 0.3])
    if goal == "maintain":
        return np.random.choice([4, 5], p=[0.6, 0.4])
    if goal == "gain":
        return np.random.choice([5, 6], p=[0.7, 0.3])
    return 4 

#df['snacks_per_day'] = df['goal_type'].map({'maintain': random.choice([1,2]), 'lose': random.choice([0,1]), 'gain': random.choice([2,3])})
dataset["snacks_per_day"] = dataset["goal_type"].apply(weighted_snacks)
#df['meals_per_day'] = df['goal_type'].map({'maintain': 3, 'lose': 3, 'gain': 5})
dataset["meals_per_day"] = dataset["goal_type"].apply(random_meals)
dataset['bodyfat_percent'] = dataset['bodyfat_percent'].fillna(0)

Case = dataset[['case_id', 'gender', 'age',  'height_cm', 'weight_kg','bodyfat_percent', 'activity_level','goal_type', 'goal_delta_kg', 'goal_time_weeks', 'plan_id']]
Plan = dataset[[ 'plan_id', 'calories_kcal', 'protein_g', 'carbs_g', 'fat_g', 'meals_per_day', 'snacks_per_day']]
#Plan.to_csv('Plan.csv', index=False)
#Case.to_csv('Case.csv', index=False)

dataset.head(10)

,case_id,gender,age,height_cm,weight_kg,bodyfat_percent,activity_level,goal_type,goal_delta_kg,goal_time_weeks,calories_kcal,protein_g,carbs_g,fat_g,meals_per_day,snacks_per_day,plan_id
0,e35dc5ec-2bc0-4b33-a347-595e9b19d7d6,male,37,167,89.8,25.7,5,maintain,0,16,3441.25,170.1,500.7,84.2,5,1,0e9db4bd-fed2-479b-b3ad-d4d92f2c9685
1,30e9ff10-3bcb-4d49-9260-21e23296739a,male,25,175,94.2,18.9,4,maintain,0,5,3484.77,194.9,500.2,78.3,4,1,c9047439-5970-4bc4-9a8a-8e744b7ad712
2,725e938d-a9c2-497c-97a0-96fe0db8269d,female,36,184,88.2,13.2,4,maintain,0,5,3490.79,160.0,523.9,83.9,4,1,1a824ade-a0bd-4028-bb46-76ef3cc40818
3,8dea4332-229c-4611-92ad-76464c65e264,female,22,180,93.4,29.6,5,gain,2,7,3832.53,176.4,531.8,111.1,6,2,54326ef7-623b-4a8a-8e3a-72d21eb2ebbf
4,539177cb-f4b2-456b-b502-97e804b3302c,female,37,184,82.5,25.9,1,maintain,0,5,2028.55,164.2,191.4,67.3,4,1,bf06a919-740d-4ea8-afae-6541de26b389
5,3d152362-454d-41d0-8736-d3e440cc4dff,female,53,173,80.1,26.5,2,maintain,0,16,2257.29,150.8,263.1,66.8,5,1,bd4ffc05-3007-49ca-81fb-517eb600ed31
6,0b66d80b-19b7-4cd8-b9aa-76d735c5d72a,female,26,186,101.2,27.6,3,lose,-7,14,2421.23,219.8,186.2,88.6,4,1,903cfcd8-9df6-4b91-8e7a-aab3c0a37ef6
7,adb81cb8-ba2f-4fee-b249-675fc35e18c1,male,21,193,96.7,25.1,1,lose,-3,8,1857.07,196.9,59.1,92.6,4,1,35ceed0f-82cd-4f18-bf96-65312c9e1ee9
8,a22334f6-9aa5-47bf-ab77-9705eae38899,female,38,156,104.9,24.7,1,maintain,0,8,2491.41,171.4,254.0,87.8,5,1,56392f68-fa0c-463d-a070-c72d89c9a626
9,c5dda57a-1feb-4ae6-941a-8ea227220faa,female,30,162,91.4,0.0,2,gain,4,13,2669.31,152.9,325.7,83.9,5,2,c5c2fb0f-1a5c-4fe0-83ed-e95ce2de4381


In [8]:
cases = pd.read_csv('Case1.csv', sep=';')
foods = pd.read_csv('Foods.csv', sep=';')
plans = pd.read_csv('Plan1.csv', sep=';')
meals = pd.read_csv('Meals.csv', sep=';')
meal_items = pd.read_csv('Meal_item_balanced.csv', sep=';')

print("💡 Betöltve:")
print("cases:", cases.shape)
print("plans:", plans.shape)
print("foods:", foods.shape)
print("meals:", meals.shape)
print("meal_items:", meal_items.shape)

cases.head()

df = cases.merge(plans, on="plan_id", how="inner")
df.drop(columns=["case_id"], inplace=True)
df.drop(columns=["plan_id"], inplace=True)

gender = {
    'male' :0,
    'female':1
}

goal_type = {
    'maintain':0,
    'lose':1,
    'gain':2
}

def weighted_snacks(goal):
    if goal == "loss":
        return np.random.choice([0, 1], p=[0.7, 0.3])
    if goal == "maintain":
        return np.random.choice([1, 2], p=[0.9, 0.1])
    if goal == "gain":
        return np.random.choice([1, 2], p=[0.2, 0.8])
    return 1 

def random_meals(goal):
    if goal == "loss":
        return np.random.choice([3, 4], p=[0.7, 0.3])
    if goal == "maintain":
        return np.random.choice([4, 5], p=[0.6, 0.4])
    if goal == "gain":
        return np.random.choice([5, 6], p=[0.7, 0.3])
    return 4 

#df['snacks_per_day'] = df['goal_type'].map({'maintain': random.choice([1,2]), 'lose': random.choice([0,1]), 'gain': random.choice([2,3])})
df["snacks_per_day"] = df["goal_type"].apply(weighted_snacks)
#df['meals_per_day'] = df['goal_type'].map({'maintain': 3, 'lose': 3, 'gain': 5})
df["meals_per_day"] = df["goal_type"].apply(random_meals)
df['gender'] = df['gender'].replace(gender)
df['goal_type'] = df['goal_type'].replace(goal_type)
df['bodyfat_percent'] = df['bodyfat_percent'].fillna(0)
df.head(20)
dfCase = df.copy()
dfCase.drop(columns=['calories_kcal', 'protein_g', 'carbs_g', 'fat_g', 'meals_per_day', 'snacks_per_day' ], inplace=True)
#dfCase.head()
dfPlan = df.copy()
dfPlan.drop(columns=['age', 'weight_kg', 'height_cm', 'gender', 'bodyfat_percent', 'goal_type', 'activity_level', 'goal_type', 'goal_delta_kg', 'goal_time_weeks'], inplace=True)
#dfPlan.head()
dfPlan.to_csv('Plan.csv', index=False)
dfCase.to_csv('Case.csv', index=False)

💡 Betöltve:
cases: (30000, 11)
plans: (30000, 7)
foods: (10000, 7)
meals: (150051, 5)
meal_items: (449906, 4)


C:\Users\Olivér\AppData\Local\Temp\ipykernel_26676\2400280700.py:53: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['gender'] = df['gender'].replace(gender)
C:\Users\Olivér\AppData\Local\Temp\ipykernel_26676\2400280700.py:54: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['goal_type'] = df['goal_type'].replace(goal_type)


In [10]:
#file_name = 'output.xlsx'
#df.to_excel(file_name)

In [11]:
# 1) cases.plan_id → plans.plan_id
missing_plan = cases[~cases["plan_id"].isin(plans["plan_id"])]
print("Cases with invalid plan_id:", len(missing_plan))

# 2) meals.plan_id → plans.plan_id
missing_plan2 = meals[~meals["plan_id"].isin(plans["plan_id"])]
print("Meals with invalid plan_id:", len(missing_plan2))

# 3) meal_items.meal_id → meals.meal_id
invalid_meal = meal_items[~meal_items["meal_id"].isin(meals["meal_id"])]
print("Meal_item with invalid meal_id:", len(invalid_meal))

# 4) meal_items.food_id → foods.food_id
invalid_food = meal_items[~meal_items["food_id"].isin(foods["food_id"])]
print("Meal_item with invalid food_id:", len(invalid_food))

Cases with invalid plan_id: 0
Meals with invalid plan_id: 0
Meal_item with invalid meal_id: 0
Meal_item with invalid food_id: 0


In [12]:
from sklearn.preprocessing import LabelEncoder



# Regressziós targetek
reg_targets = [
    "calories_kcal",
    "protein_g",
    "carbs_g",
    "fat_g",
    "meals_per_day",
    "snacks_per_day"
]

Y = df[reg_targets]


In [13]:
# User input feature-ek
base_features = [
    "gender",
    "age",
    "height_cm",
    "weight_kg",
    "bodyfat_percent",
    "activity_level",
    "goal_type",
    "goal_delta_kg",
    "goal_time_weeks"
]

X = df[base_features]

In [14]:
x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42,)
model_rfr = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=None, n_jobs=-1)
model_rfr.fit(x_train, y_train)
y_pred = model_rfr.predict(x_test)


rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Overall RMSE:", rmse)

for i, col in enumerate(reg_targets):
    r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"R2 for {col}: {r2:.3f}")




Overall RMSE: 25.572507558279597
R2 for calories_kcal: 0.994
R2 for protein_g: 0.900
R2 for carbs_g: 0.976
R2 for fat_g: 0.849
R2 for meals_per_day: 0.636
R2 for snacks_per_day: 0.568


In [ ]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Alap XGBoost modell
xgb_base = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    tree_method="hist", 
    random_state=42
)

model = MultiOutputRegressor(xgb_base)

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Overall RMSE:", rmse)
for i, col in enumerate(reg_targets):
    r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"R2 for {col}: {r2:.3f}")


Overall RMSE: 18.91328881756117
R2 for calories_kcal: 0.997
R2 for protein_g: 0.908
R2 for carbs_g: 0.981
R2 for fat_g: 0.858
R2 for meals_per_day: 0.649
R2 for snacks_per_day: 0.577


In [16]:
import joblib
# Modell mentése

joblib.dump(model, 'multioutput_xgb_model.pkl')
print("Modell elmentve: multioutput_xgb_model.pkl")

Modell elmentve: multioutput_xgb_model.pkl
